## 1. 탐색적 데이터 분석(EDA) 및 전처리 파이프라인 구현

### [문항 1-1] 데이터 구조 탐색 및 기초 분석

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('diabetes.csv') 

print("[데이터 구조]")
print("=" * 70)
print("행렬 크기:",df.shape)
print("컬럼별 자료형 및 결측치 개수 확인:")
df.info()

[데이터 구조]
행렬 크기: (768, 9)
컬럼별 자료형 및 결측치 개수 확인:
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [2]:
print("=" * 70)
print("컬럼별 이상치 존재 여부 확인:\n",df.describe())

컬럼별 이상치 존재 여부 확인:
        Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.00000

- Insulin, SkinThickness 에서 이상치 존재 확인
- 또한, describe()로 이상치가 있는지 확인하던 도중에 [Glucose, BloodPressure, SkinThickness, Insulin, BMI] 컬럼에서 min 값이 0으로 들어가 있음을 확인
- 위 컬럼들은 값이 0이면 실제 생리학적으로 의미가 없거나, 매우 비현실적인 값임. (사람의 혈압이 0일 수는 없음)
- Pregnancies(임신 횟수)나 Outcome(당뇨 여부)는 0이 정상적인 값임.
- 따라서, 위 5개 컬럼에서 0은 결측치로서 기록된 값으로 판단함.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI'] # 5개 컬럼을 zero_cols로 묶어 컬럼벌 0값 확인

print("상위 15개 데이터의 0값 확인:")
print(df[zero_cols].head(15))

print("컬럼별 0값(결측치) 개수:")
print((df[zero_cols] == 0).sum())

### [문항 1-2] 전처리 파이프라인 구축

#### 분석된 결과를 바탕으로 결측치 및 이상치 처리

- 0을 NaN으로 바꾸는 것을 결측치 대체 방식으로 선택함

In [ ]:
# 결측치 및 이상치 처리
# 0으로 입력된 결측치를 NaN으로 바꾼다
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)
print("NaN으로 변경된 값 확인:")
print(df.isna().sum())

In [ ]:
# Glucose, BloodPressure, BMI
# → 평균값을 사용하여 결측치 대체
df['Glucose'] = df['Glucose'].fillna(df['Glucose'].mean())
df['BloodPressure'] = df['BloodPressure'].fillna(df['BloodPressure'].mean())
df['BMI'] = df['BMI'].fillna(df['BMI'].mean())

# Insulin, SkinThickness
# → 이상치의 영향을 줄이기 위해 중앙값으로 결측치 대체
df['Insulin'] = df['Insulin'].fillna(df['Insulin'].median())
df['SkinThickness'] = df['SkinThickness'].fillna(df['SkinThickness'].median())

print("컬럼별 총 결측치 개수 확인:")
print(df.isna().sum())

#### 데이터 스케일러(StandardScaler) 적용 및 선택 근거
- 머신러닝 학습 전, 변수마다 값의 범위와 분산이 다르므로 스케일 차이를 줄이기 위해 StandardScaler를 이용하여 평균 0과 표준편차 1을 기준으로 데이터를 표준화한다.
- 데이터 누수(Data Leakage)를 방지하기 위해 Train/Test Split 이후 Train 데이터에만 fit하고, Test 데이터에는 동일한 기준으로 transform한다.

**X에는 범주형 변수가 존재하지 않으므로, 범주형 변수 인코딩을 수행하지 않음**

In [ ]:
# X, y 분리
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Train, Test 분리 -> 데이터 누수 방지
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 데이터 스케일링
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("전처리 완료")
print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)

### [문항 1-2] 데이터 시각화 및 인사이트 도출

[통계 인사이트]

1. 히스토그램: 당뇨 환자 연령 분포
- 대상자의 나이는 20~70세 분포
- 20~40대 비중이 가장 높으며, 50대부터 점차 감소

2. 박스플롯: 당뇨 여부에 따른 BMI 분포 비교
   - 당뇨 환자 그룹: x축 Outcome=1, 당뇨가 없는 그룹: x축 Outcome=0
   - 당뇨 환자 그룹에서 BMI 중앙값이 더 높은 것으로 보아, BMI가 높을수록 당뇨가 있는 경향이 있음.
   - BMI 50 이상에서 이상치도 확인 가능함.
   - 당뇨 환자에서 BMI가 60이 현저히 넘는 극단적 이상치도 확인 가능함.

3. 산점도: 비만과 혈당 및 당뇨의 관계 표현
   - 당뇨 환자 그룹: 푸른색 점, 비환자 그룹: 빨간색 점
   - x축: 비만도(BMI), y축: 혈당 수치(Glucose Level)
   - 당뇨 환자 그룹에서 높은 혈당값을 가진 데이터가 비당뇨 환자에 비해 상대적으로 많이 나타남
   - BMI와 혈당의 뚜렷한 선형관계는 나타나지 않음. 
   - Glucose가 BMI보다 당뇨 여부를 구분하는 데 더 강한 변수임을 확인할 수 있음.

In [ ]:
# 3개 그래프(히스토그램, 박스플롯, 산점도)를 위한 도화지와 액자 준비
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 당뇨가 있는 사람들만 모으기 
df_diabetes = df[df['Outcome'] == 1]

# 히스토그램: 당뇨 나이 분포 - 당뇨 환자의 연령 분포 확인
axes[0].hist(df_diabetes['Age'], bins=10, color='red', edgecolor='black')
axes[0].set_title('Age Distribution of Diabetic Patients')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# 박스플롯: 당뇨 여부에 따른 BMI 분포 비교
sns.boxplot(x='Outcome', y='BMI', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('BMI by Diabetes Status')
axes[1].set_xlabel('Outcome')
axes[1].set_ylabel('BMI')

# 산점도: 비만과 혈당 및 당뇨의 관계
scatter = axes[2].scatter(
    df['BMI'],
    df['Glucose'],
    c=df['Outcome'],
    cmap='coolwarm',
    alpha=0.7
)
axes[2].set_title('BMI vs. Glucose by Diabetes Status')
axes[2].set_xlabel('Body Mass Index (BMI, kg/m²)')
axes[2].set_ylabel('Glucose Level (mg/dL)')

fig.colorbar(scatter, ax=axes[2], label='Diabetes Outcome (0: Negative, 1: Positive)')

plt.tight_layout()
plt.show()

## 2. 머신러닝 모델 학습, 과적합 점검 및 하이퍼파라미터 튜닝

### 필수 구현 내용

In [ ]:
# [문항 2-1] 회귀 및 분류 모델 학습과 성능 측정

전처리가 완료된 데이터를 사용하여 과제 목적에 맞는 회귀 모델과 분류 모델을 각각 정의하여 학습시키고, 각 모델에 부합하는 평가지표(MSE, $R^2$, Accuracy, F1-score 등)를 활용하여 모델의 성능을 측정합니다.

[문항 2-1 결과물]: 회귀 및 분류 모델 학습 소스 코드와 선정된 평가지표의 최종 성능 수치 출력 화면

### 심화 구현 내용

In [ ]:
# [문항 2-2] 평가지표 선정 근거 및 예측 결과 해석

각 과제 유형에서 해당 평가지표를 모델 평가 기준으로 선택한 이유를 명시하고, 최종 측정된 성능 수치에 대한 분석 및 예측 오류가 발생한 원인에 대한 해석을 깊이 있게 기술합니다.

[문항 2-2 결과물]: 평가지표 선택 이유, 모델 예측 성능에 대한 분석 및 오류 원인 해석 내용 (마크다운 셀)

In [ ]:
# [문항 2-3] 규제, 교차 검증, 하이퍼파라미터 튜닝을 통한 성능 개선

모델의 과적합(Overfitting)을 방지하고 성능을 고도화하기 위해 규제(Regularization), 교차 검증(Cross-Validation), 하이퍼파라미터 튜닝(GridSearchCV 등)의 3가지 요소를 모두 적용하여 모델을 개선하고, 개선 전후의 성능을 비교 분석합니다.

[문항 2-3 결과물]: 규제, 교차 검증, 튜닝이 반영된 모델 개선 코드 및 개선 전후의 성능 지표 비교 테이블과 기법별 효과 요약 글 (마크다운 셀)

## 3. Hugging Face 사전 학습된 모델을 활용한 뉴스 기사 분류 및 추론

### 필수 구현 내용

In [ ]:
# [문항 3-1] 사전 학습된 모델 및 토크나이저 로드

In [ ]:
# [문항 3-2] 문맥 내 학습(In-context Learning) 기반 뉴스 기사 분류 추론

### 심화 구현 내용

In [ ]:
# [문항 3-3] Generation 하이퍼파라미터 변경 실험 및 성능 비교 해석